# Colab 3: Combined Two-Dataset Training

This notebook clones the repo, installs dependencies, uses two Google Drive dataset folders, trains the combined model, evaluates it, zips the outputs, copies the zip to Drive, and downloads it.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
REPO_URL = 'https://github.com/au510621104021/FND_2027.git'
BRANCH = 'main'
PROJECT_DIR = '/content/FND_2027'

# Default Drive dataset folders
DATASET_DIR_1 = '/content/drive/MyDrive/ISOT Fake News Dataset'
DATASET_DIR_2 = '/content/drive/MyDrive/Fake News Dataset'

# Output zip copy destination in Drive
DRIVE_OUTPUT_DIR = '/content/drive/MyDrive/FND_2027_outputs'

In [ ]:
import os
from pathlib import Path

os.chdir('/content')
!rm -rf /content/FND_2027
!git clone --branch main https://github.com/au510621104021/FND_2027.git /content/FND_2027

assert Path(PROJECT_DIR).exists(), f'Project directory not found: {PROJECT_DIR}'
os.chdir(PROJECT_DIR)
print('Project directory:', os.getcwd())
!ls

In [ ]:
import os
from pathlib import Path

os.chdir(PROJECT_DIR)
assert Path('requirements.txt').exists(), 'requirements.txt not found. Clone step likely failed.'
!pip install -q -r requirements.txt

In [ ]:
from pathlib import Path

def require_dataset_dir(path_str):
    path = Path(path_str)
    if not path.exists():
        raise FileNotFoundError(f'Folder not found: {path_str}')
    csvs = list(path.glob('*.csv'))
    if not csvs:
        raise FileNotFoundError(f'No CSV files found in: {path_str}')
    print(f'OK: {path_str}')
    print('CSV files:', [p.name for p in csvs[:10]])
    return str(path)

DATASET_DIR_1 = require_dataset_dir(DATASET_DIR_1)
DATASET_DIR_2 = require_dataset_dir(DATASET_DIR_2)

In [ ]:
import os
import yaml
from pathlib import Path

os.chdir(PROJECT_DIR)

config_path = Path('config/config.yaml')
with config_path.open('r', encoding='utf-8') as f:
    config = yaml.safe_load(f)

config['data']['dataset_name'] = 'generic'
config['data']['data_dir'] = DATASET_DIR_1
config['data']['dataset_names'] = ['generic', 'generic']
config['data']['data_dirs'] = [DATASET_DIR_1, DATASET_DIR_2]
config['logging']['checkpoint_dir'] = './checkpoints'
config['logging']['log_dir'] = './logs'
config['inference']['model_checkpoint'] = './checkpoints/best_model_combined.pt'

with config_path.open('w', encoding='utf-8') as f:
    yaml.safe_dump(config, f, sort_keys=False)

print('Updated config for combined training:')
print(config['data'])

In [ ]:
import torch
print('CUDA available:', torch.cuda.is_available())
print('GPU name:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

In [ ]:
import os
os.chdir(PROJECT_DIR)
!python scripts/train.py --config config/config.yaml

In [ ]:
import os
import shutil
from pathlib import Path

os.chdir(PROJECT_DIR)
best_model = Path('checkpoints/best_model.pt')
combined_model = Path('checkpoints/best_model_combined.pt')
assert best_model.exists(), 'Training did not produce checkpoints/best_model.pt'
shutil.copy2(best_model, combined_model)
print(f'Saved combined checkpoint to: {combined_model}')

In [ ]:
import os
os.chdir(PROJECT_DIR)
!python scripts/evaluate.py --config config/config.yaml --checkpoint checkpoints/best_model_combined.pt

In [ ]:
import os
import shutil
from pathlib import Path

os.chdir(PROJECT_DIR)
zip_base = Path('combined_model_artifacts')
zip_path = shutil.make_archive(str(zip_base), 'zip', root_dir='.', base_dir='checkpoints')
print(f'Created zip: {zip_path}')
!zip -ur combined_model_artifacts.zip results logs -x "*.ipynb_checkpoints*"

In [ ]:
import os
from pathlib import Path
from google.colab import files

os.chdir(PROJECT_DIR)
zip_path = Path('combined_model_artifacts.zip')
assert zip_path.exists(), 'Zip file was not created.'

os.makedirs(DRIVE_OUTPUT_DIR, exist_ok=True)
!cp combined_model_artifacts.zip "$DRIVE_OUTPUT_DIR/"
print(f'Copied zip to Drive: {DRIVE_OUTPUT_DIR}')

files.download(str(zip_path))